# bronze layer - raw smart meter data ingestion

this notebook is responsible for loading the raw smart meter readings into the bronze layer.
the data is stored as delta format without any transformations so that the original records are always available for auditing and reprocessing.

In [0]:
# importing the spark functions we'll use in this notebook

from pyspark.sql import functions as f

In [0]:
# keeping the input file path in one place makes it easier to maintain

meter_readings_path = "/Volumes/smart_meter_project_catalog/default/meter_problem_project_volume/meter_readings.csv"

In [0]:
# reading the raw meter readings from the volume

bronze_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(meter_readings_path)
)

In [0]:
# taking a quick look at the data

display(bronze_df)

meter_id,household_id,timestamp,units_consumed
M001,H001,2026-04-01T00:00:00.000Z,0.41
M002,H002,2026-04-01T00:00:00.000Z,0.1
M003,H003,2026-04-01T00:00:00.000Z,0.44
M004,H004,2026-04-01T00:00:00.000Z,0.23
M005,H005,2026-04-01T00:00:00.000Z,6.93
M006,H006,2026-04-01T00:00:00.000Z,0.8
M007,H007,2026-04-01T00:00:00.000Z,0.47
M008,H008,2026-04-01T00:00:00.000Z,0.2
M009,H009,2026-04-01T00:00:00.000Z,0.25
M010,H010,2026-04-01T00:00:00.000Z,0.25


In [0]:
# checking whether spark inferred the correct data types

bronze_df.printSchema()


# adding ingestion information for tracking purposes

bronze_df = (
    bronze_df
        .withColumn("ingestion_time", f.current_timestamp())
        .withColumn("ingestion_date", f.current_date())
)

# checking the new columns

display(bronze_df)

root
 |-- meter_id: string (nullable = true)
 |-- household_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- units_consumed: double (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- ingestion_date: date (nullable = false)



meter_id,household_id,timestamp,units_consumed,ingestion_time,ingestion_date
M001,H001,2026-04-01T00:00:00.000Z,0.41,2026-07-10T18:06:17.618Z,2026-07-10
M002,H002,2026-04-01T00:00:00.000Z,0.1,2026-07-10T18:06:17.618Z,2026-07-10
M003,H003,2026-04-01T00:00:00.000Z,0.44,2026-07-10T18:06:17.618Z,2026-07-10
M004,H004,2026-04-01T00:00:00.000Z,0.23,2026-07-10T18:06:17.618Z,2026-07-10
M005,H005,2026-04-01T00:00:00.000Z,6.93,2026-07-10T18:06:17.618Z,2026-07-10
M006,H006,2026-04-01T00:00:00.000Z,0.8,2026-07-10T18:06:17.618Z,2026-07-10
M007,H007,2026-04-01T00:00:00.000Z,0.47,2026-07-10T18:06:17.618Z,2026-07-10
M008,H008,2026-04-01T00:00:00.000Z,0.2,2026-07-10T18:06:17.618Z,2026-07-10
M009,H009,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:06:17.618Z,2026-07-10
M010,H010,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:06:17.618Z,2026-07-10


In [0]:
# counting the total number of records received

print(f"total records loaded : {bronze_df.count()}")

total records loaded : 1400


In [0]:
# creating the catalog if it does not already exist

spark.sql("""
create catalog if not exists smart_meter_project_catalog
""")

DataFrame[]

In [0]:
spark.sql("""
use catalog smart_meter_project_catalog
""")


# creating the bronze schema

spark.sql("""
create schema if not exists bronze
""")


# saving the raw data into the bronze layer as a delta table
# partitioning by ingestion_date helps spark read data faster later

(
    bronze_df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("ingestion_date")
        .saveAsTable("bronze.meter_readings")
)

# checking whether the table was created successfully

display(
    spark.table("bronze.meter_readings")
)

meter_id,household_id,timestamp,units_consumed,ingestion_time,ingestion_date
M001,H001,2026-04-01T00:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10
M002,H002,2026-04-01T00:00:00.000Z,0.1,2026-07-10T18:19:52.606Z,2026-07-10
M003,H003,2026-04-01T00:00:00.000Z,0.44,2026-07-10T18:19:52.606Z,2026-07-10
M004,H004,2026-04-01T00:00:00.000Z,0.23,2026-07-10T18:19:52.606Z,2026-07-10
M005,H005,2026-04-01T00:00:00.000Z,6.93,2026-07-10T18:19:52.606Z,2026-07-10
M006,H006,2026-04-01T00:00:00.000Z,0.8,2026-07-10T18:19:52.606Z,2026-07-10
M007,H007,2026-04-01T00:00:00.000Z,0.47,2026-07-10T18:19:52.606Z,2026-07-10
M008,H008,2026-04-01T00:00:00.000Z,0.2,2026-07-10T18:19:52.606Z,2026-07-10
M009,H009,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:19:52.606Z,2026-07-10
M010,H010,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:19:52.606Z,2026-07-10


In [0]:
spark.sql("""
select *
from bronze.meter_readings
limit 10
""").display()

meter_id,household_id,timestamp,units_consumed,ingestion_time,ingestion_date
M001,H001,2026-04-01T00:00:00.000Z,0.41,2026-07-10T18:19:52.606Z,2026-07-10
M002,H002,2026-04-01T00:00:00.000Z,0.1,2026-07-10T18:19:52.606Z,2026-07-10
M003,H003,2026-04-01T00:00:00.000Z,0.44,2026-07-10T18:19:52.606Z,2026-07-10
M004,H004,2026-04-01T00:00:00.000Z,0.23,2026-07-10T18:19:52.606Z,2026-07-10
M005,H005,2026-04-01T00:00:00.000Z,6.93,2026-07-10T18:19:52.606Z,2026-07-10
M006,H006,2026-04-01T00:00:00.000Z,0.8,2026-07-10T18:19:52.606Z,2026-07-10
M007,H007,2026-04-01T00:00:00.000Z,0.47,2026-07-10T18:19:52.606Z,2026-07-10
M008,H008,2026-04-01T00:00:00.000Z,0.2,2026-07-10T18:19:52.606Z,2026-07-10
M009,H009,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:19:52.606Z,2026-07-10
M010,H010,2026-04-01T00:00:00.000Z,0.25,2026-07-10T18:19:52.606Z,2026-07-10
